In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import CocoDetection
from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO

In [2]:
class CocoSegmentationDataset(Dataset):
    def __init__(self, img_folder, ann_file, processor):
        self.img_folder = img_folder
        self.coco = COCO(ann_file)
        self.img_ids = self.coco.getImgIds()
        self.processor = processor

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        annotations = self.coco.loadAnns(ann_ids)
        image_info = self.coco.loadImgs(img_id)[0]
    
        image = Image.open(f"{self.img_folder}/{image_info['file_name']}").convert("RGB")
    
        mask = np.zeros((image_info["height"], image_info["width"]), dtype=np.uint8)
        for ann in annotations:
            ann_mask = self.coco.annToMask(ann)
            mask[ann_mask == 1] = ann['category_id']
    
        inputs = self.processor(images=image, segmentation_maps=mask, return_tensors="pt")
    
        # Handle mask_labels and class_labels explicitly
        inputs = {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "mask_labels": inputs["mask_labels"][0],
            "class_labels": inputs["class_labels"][0]
        }
    
        return inputs

def custom_collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])

    mask_labels = [item["mask_labels"] for item in batch]
    class_labels = [item["class_labels"] for item in batch]

    # Pad mask_labels and class_labels to the maximum number of instances in the batch
    max_num_masks = max(m.shape[0] for m in mask_labels)
    padded_mask_labels = []
    padded_class_labels = []

    for masks, classes in zip(mask_labels, class_labels):
        num_masks = masks.shape[0]

        # Padding masks
        pad_size = (0, 0, 0, 0, 0, max_num_masks - num_masks)
        padded_masks = torch.nn.functional.pad(masks, pad_size, value=0)
        padded_mask_labels.append(padded_masks)

        # Padding class labels
        padded_classes = torch.cat([classes, torch.zeros(max_num_masks - num_masks, dtype=classes.dtype)])
        padded_class_labels.append(padded_classes)

    padded_mask_labels = torch.stack(padded_mask_labels)
    padded_class_labels = torch.stack(padded_class_labels)

    return {
        "pixel_values": pixel_values,
        "mask_labels": padded_mask_labels,
        "class_labels": padded_class_labels
    }


processor = Mask2FormerImageProcessor.from_pretrained("facebook/mask2former-swin-base-coco-instance")
model = Mask2FormerForUniversalSegmentation.from_pretrained(
    "facebook/mask2former-swin-base-coco-instance",
    ignore_mismatched_sizes=True
)

train_dataset = CocoSegmentationDataset('C:/Users/mza0288/Datset/train2017', 'C:/Users/mza0288/Datset/annotations/instances_train2017.json', processor)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=custom_collate_fn)

C:\Users\mza0288\AppData\Local\anaconda3\envs\torchenv\lib\site-packages\transformers\utils\deprecation.py:165: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size'
  return func(*args, **kwargs)


loading annotations into memory...
Done (t=0.03s)
creating index...
index created!


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

model.train()
epochs = 10

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        pixel_values = batch["pixel_values"].to(device)
        mask_labels = batch["mask_labels"].to(device)
        class_labels = batch["class_labels"].to(device)

        outputs = model(pixel_values=pixel_values, mask_labels=mask_labels, class_labels=class_labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")

KeyboardInterrupt: 